# Magic Hour · Krea 2 Identity Edit

Internal production demo for deterministic head / face swap.

| Input | Role |
| --- | --- |
| **Body** | Scene — pose, clothing, background |
| **Face** | Identity — face / head to transfer |

**Runtime:** GPU → prefer **A100**. T4 works but is slower (~3–4 min/image).

**How to run:** set knobs in §1 → **Runtime → Run all**.  
Warm re-runs: change knobs / re-upload if needed → re-run **§6 only**.

Weights cache on Drive so reconnects do not re-download ~18 GB.

---
## 1 · Settings

Only these knobs are meant to change. Everything else uses production defaults.

In [ ]:
# User-facing knobs (edit these)
SEED = 46
PROMPT = None          # None = use yaml default prompt
STEPS = 8
CFG = 1.0
OUTPUT_LONG_SIDE = 1024   # final body canvas long side (px)
STITCH = True             # mask→crop→edit→soft stitch (recommended)
DEBUG = False             # True: verbose logs + debug_*.png intermediates

print("Settings")
print(f"  SEED={SEED}  STEPS={STEPS}  CFG={CFG}  OUTPUT_LONG_SIDE={OUTPUT_LONG_SIDE}")
print(f"  STITCH={STITCH}  DEBUG={DEBUG}")
if PROMPT:
    print(f"  PROMPT={PROMPT[:80]}…" if len(str(PROMPT)) > 80 else f"  PROMPT={PROMPT}")
else:
    print("  PROMPT=(yaml default)")

---
## 2 · GPU check

In [ ]:
from pathlib import Path

assert Path("/content").exists(), (
    "This notebook is for Google Colab (/content)."
)

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime → Change runtime type → GPU (prefer A100), "
        "then Runtime → Run all."
    )

print(f"✓ GPU  {torch.cuda.get_device_name(0)}")
print(f"  VRAM {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
print(f"  Torch {torch.__version__}")

---
## 3 · Setup (Drive · repo · ComfyUI · models)

First session: mount Drive, clone, install, download (~18 GB once).  
Later sessions: verifies weights on Drive; may briefly reinstall ComfyUI on a fresh runtime.

> After the **first** custom-node install in a brand-new runtime, restart once  
> (**Runtime → Restart session**), then **Run all** again from §1.

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

from google.colab import drive

print("→ Mounting Google Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
print("→ Syncing repository…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

spec = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

print("→ Installing ComfyUI + Krea2 nodes (idempotent)…")
!bash scripts/setup_colab.sh
!bash scripts/setup_krea2_nodes.sh

print("→ Downloading / verifying models…")
!python scripts/download_krea2.py \
  --comfy "$COMFYUI_PATH" \
  --store-dir "$HEADSWAP_MODEL_STORE" \
  --staging-dir "$HEADSWAP_STAGING_DIR" \
  --backend auto \
  --disable-xet

spec_demo = importlib.util.spec_from_file_location(
    "colab_demo", REPO / "scripts" / "colab_demo.py"
)
colab_demo = importlib.util.module_from_spec(spec_demo)
spec_demo.loader.exec_module(colab_demo)
colab_demo.verify_models(PATHS["model_store"])
colab_demo.ok(f"Setup ready — models at {PATHS['model_store']}")
print(
    "\nIf this was the FIRST custom-node install on a fresh runtime: "
    "Runtime → Restart session, then Run all from §1.\n"
)

---
## 4 · Upload body & face

Run this cell, then choose **body** (scene) then **face** (identity).  
JPG / PNG / WEBP are fine. Faces are validated before continuing.

In [ ]:
import importlib.util
from pathlib import Path

from google.colab import files
from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location(
    "colab_demo", REPO / "scripts" / "colab_demo.py"
)
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
cache_dir = REPO / ".cache" / "headswap_v2"
cache_dir.mkdir(parents=True, exist_ok=True)

try:
    print("Upload BODY image (scene / clothing / pose)…")
    up_body = files.upload()
    if not up_body:
        raise colab_demo.DemoError("No body image uploaded.")
    body_name = next(iter(up_body))
    body_path = custom / "body.png"
    body_im = colab_demo.save_upload(up_body[body_name], body_path)
    body_face = colab_demo.require_face(body_im, cache_dir, "body")

    print("Upload FACE image (identity donor)…")
    up_face = files.upload()
    if not up_face:
        raise colab_demo.DemoError("No face image uploaded.")
    face_name = next(iter(up_face))
    face_path = custom / "face.png"
    face_im = colab_demo.save_upload(up_face[face_name], face_path)
    face_face = colab_demo.require_face(face_im, cache_dir, "face")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))

!python scripts/prepare_eval_set.py --custom data/custom

display(Markdown("### Inputs"))
print(f"Body face conf={body_face['confidence']}  size={body_im.size}")
display(body_im.resize((320, 320)))
print(f"Face face conf={face_face['confidence']}  size={face_im.size}")
display(face_im.resize((320, 320)))
colab_demo.ok(f"Saved → {body_path} , {face_path}")

---
## 5 · Preflight

Confirms GPU, model files, and detectable faces before spending GPU time.

In [ ]:
import importlib.util
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec = importlib.util.spec_from_file_location(
    "colab_demo", REPO / "scripts" / "colab_demo.py"
)
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

try:
    PREFLIGHT = colab_demo.preflight(
        body_path=REPO / "data" / "custom" / "body.png",
        face_path=REPO / "data" / "custom" / "face.png",
        cache_dir=REPO / ".cache" / "headswap_v2",
        model_store=PATHS["model_store"],
    )
    colab_demo.environment_summary(
        paths=PATHS,
        seed=SEED,
        stitch=STITCH,
        debug=DEBUG,
    )
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))
except NameError:
    colab_demo.fail("Run §1 Settings first (SEED / STITCH / DEBUG).")
    raise SystemExit("Run §1 Settings first.")

---
## 6 · Run inference

Models load once and stay warm. Re-run this cell for another image or knob change.

In [ ]:
import importlib.util
import json
import time
import traceback
from pathlib import Path

from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location(
    "colab_env", REPO / "scripts" / "colab_env.py"
)
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec = importlib.util.spec_from_file_location(
    "colab_demo", REPO / "scripts" / "colab_demo.py"
)
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

CONFIG = REPO / "configs" / "krea2_identity_edit.yaml"
OUT = REPO / "results" / "krea2_identity_edit"
PAIR_DIR = OUT / "images" / "custom_001"
PAIR_DIR.mkdir(parents=True, exist_ok=True)
BODY_PATH = REPO / "data" / "custom" / "body.png"
FACE_PATH = REPO / "data" / "custom" / "face.png"

RUN_OK = False
RUN_ERROR = None
RESULT_PATH = None
STABLE_PATH = None
META = {}

try:
    if "SEED" not in globals():
        raise colab_demo.DemoError("Run §1 Settings first.")

    colab_demo.preflight(
        body_path=BODY_PATH,
        face_path=FACE_PATH,
        cache_dir=REPO / ".cache" / "headswap_v2",
        model_store=PATHS["model_store"],
    )

    cfg = colab_demo.apply_user_knobs(
        load_config(CONFIG),
        seed=SEED,
        prompt=PROMPT,
        steps=STEPS,
        cfg_scale=CFG,
        output_long_side=OUTPUT_LONG_SIDE,
        stitch=STITCH,
        debug=DEBUG,
    )

    # Warm path: hold one pipeline / shared Comfy runtime across re-runs.
    if "_KREA2_PIPE" not in globals() or _KREA2_PIPE is None:
        colab_demo.progress("Loading models (cold start)…")
        rt = get_shared_krea2_runtime(init_custom_nodes=True)
        _KREA2_PIPE = create_pipeline(cfg, runtime=rt)
    else:
        colab_demo.progress("Reusing warm model cache…")
        _KREA2_PIPE.cfg.update(cfg)

    body = Image.open(BODY_PATH).convert("RGB")
    face = Image.open(FACE_PATH).convert("RGB")

    colab_demo.progress("Running head swap…")
    t0 = time.perf_counter()
    if DEBUG:
        result = _KREA2_PIPE.run(body, face, out_dir=PAIR_DIR)
    else:
        with colab_demo.quiet_logs(debug=False):
            result = _KREA2_PIPE.run(body, face, out_dir=PAIR_DIR)
    wall = colab_demo.elapsed(t0)

    RESULT_PATH = PAIR_DIR / "result.png"
    result.image.save(RESULT_PATH)
    STABLE_PATH = colab_demo.write_stable_result(RESULT_PATH, PATHS["outputs"])

    META = dict(result.meta or {})
    metrics = {
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "config": str(CONFIG),
        "pipeline": META.get("pipeline", "krea2_identity_edit"),
        "n_pairs": 1,
        "pairs": [
            {
                "pair_id": "custom_001",
                "success": True,
                "latency_s": float(result.latency_s),
                "result_path": str(RESULT_PATH),
                "stable_path": str(STABLE_PATH),
                "meta": META,
                "debug_paths": dict(result.debug_paths or {}),
            }
        ],
    }
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))

    timing = META.get("timing_s") or {}
    colab_demo.print_run_summary(
        success=True,
        total_s=wall,
        sampling_s=timing.get("diffusion_sampling"),
        steps=META.get("steps"),
        seed=META.get("seed"),
        gpu=(PREFLIGHT.get("gpu") or {}).get("name") if "PREFLIGHT" in globals() else None,
        resolution=result.image.size,
        output_path=STABLE_PATH,
        cache_hit=META.get("model_cache_hit"),
    )
    RUN_OK = True

except colab_demo.DemoError as exc:
    RUN_ERROR = str(exc)
    colab_demo.print_run_summary(
        success=False,
        total_s=0.0,
        sampling_s=None,
        steps=None,
        seed=globals().get("SEED"),
        gpu=None,
        resolution=None,
        output_path=None,
        error=RUN_ERROR,
    )
except Exception as exc:
    RUN_ERROR = str(exc)
    colab_demo.fail(f"Unexpected failure: {exc}")
    if globals().get("DEBUG"):
        traceback.print_exc()
    else:
        print("Set DEBUG=True in §1 and re-run for a full traceback.")
    colab_demo.print_run_summary(
        success=False,
        total_s=0.0,
        sampling_s=None,
        steps=None,
        seed=globals().get("SEED"),
        gpu=None,
        resolution=None,
        output_path=None,
        error=RUN_ERROR,
    )

---
## 7 · Results

Preview + download. Debug intermediates only appear when `DEBUG=True`.

In [ ]:
import importlib.util
from pathlib import Path

from google.colab import files
from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location(
    "colab_demo", REPO / "scripts" / "colab_demo.py"
)
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

if not globals().get("RUN_OK"):
    colab_demo.fail(globals().get("RUN_ERROR") or "No successful run yet — fix §6 and re-run.")
else:
    body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
    face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
    result = Image.open(RESULT_PATH).convert("RGB")

    display(Markdown("### Body · Face · Result"))
    display(colab_demo.show_side_by_side(body, face, result))

    if globals().get("DEBUG"):
        display(Markdown("### Debug intermediates"))
        shown = 0
        for name in [
            "debug_face_crop.png",
            "debug_crop.png",
            "debug_mask.png",
            "debug_edited_crop.png",
            "debug_body.png",
            "debug_person.png",
        ]:
            p = Path(RESULT_PATH).parent / name
            if p.exists():
                print(name)
                display(Image.open(p).convert("RGB").resize((256, 256)))
                shown += 1
        if not shown:
            print("(no debug_*.png — pipeline save_debug may be off)")

    download_path = Path(STABLE_PATH) if STABLE_PATH else Path(RESULT_PATH)
    display(Markdown("### Download"))
    print(download_path)
    try:
        files.download(str(download_path))
    except Exception as exc:
        print(f"Browser download helper failed ({exc}). File is at: {download_path}")

---
## Notes

| Topic | Behavior |
| --- | --- |
| Determinism | Fixed `SEED` (default 46); same inputs → same output |
| Warm cache | Re-run §6 only; `model_cache_hit` should become True |
| Outputs | `/content/headswap_outputs/HEADSWAP_RESULT.png` |
| Stitch | `STITCH=True` preserves body/BG outside the head |
| Debug | `DEBUG=True` enables verbose logs + `debug_*.png` |
| Docs | `COLAB.md` |

*Magic Hour · headswap_V2 internal demo*